# 🕵️ Dia 75 — Documentação do caso: da evidência preservada à linha do tempo

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 75* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-75.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Sexta, 30/out. Sete arquivos do Seu Antônio na mesa, e você já abriu o extrato no editor "só para dar uma olhada". Rafa: *"Quantos arquivos? Sete. Quantos hashes?"* Silêncio.

*"E o extrato está em UTC, o celular dele em horário de Brasília, e o print não diz fuso nenhum."*

Neste notebook você vai:

1. **Criar os originais** (os 5 do caso do original e os 7 do Seu Antônio) e calcular os SHA-256 na coleta.
2. **Converter cada hora local para UTC** com `zoneinfo`.
3. **Montar a linha do tempo** e conferir os achados do Checkpoint.
4. **Estragar um original de propósito** e ver a custódia registrar a falha.
5. **Jogar o quiz "confirmado, provável ou contestado?"**.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo aqui é fictício (Operação Isca).

## 1. Os originais e os hashes da coleta

Criamos as duas pastas de casos, exatamente com o conteúdo dos arquivos (o do original em inglês, para o seu hash bater com o de qualquer outro aluno). Depois gravamos `hashes-at-collection.txt` no **mesmo formato do `shasum -a 256`** (`<hash>  <arquivo>`), para você poder conferir com a ferramenta do sistema também.

> Na vida real você **não cria** os originais: eles vêm da vítima. Aqui criamos para ter algo a preservar, como o original manda.

In [ ]:
import hashlib                      # SHA-256
import shutil                       # copiar originals/ → working/
from datetime import datetime, timezone
from pathlib import Path

# Caso do original (fictício, em inglês, exatamente como no Day 75)
ORIGINAIS_FICT = {
    'E01-first-text.txt': 'SMS from +1 202 555 0147, received 2026-01-06 19:42 (phone set to America/Chicago): Hi Lena, is the dinner still on for Friday?\n',
    'E02-app-screenshot-note.txt': 'Screenshot description: aurum-desk.example dashboard showing balance $1,240.00, phone clock 2026-01-24 08:15 America/Chicago.\n',
    'E03-exchange-record.csv': 'time_utc,amount_usdt,to_address,tx_hash\n2026-01-22T15:03:11Z,1000,TFICT-A1,0xFICT01\n2026-02-03T16:40:55Z,5000,TFICT-A1,0xFICT02\n2026-02-10T17:02:09Z,4000,TFICT-A1,0xFICT03\n2026-02-18T15:20:47Z,6000,TFICT-B7,0xFICT04\n2026-02-26T19:11:30Z,3000,TFICT-B7,0xFICT05\n',
    'E04-withdrawal-message.txt': 'In-app message 2026-03-01 10:05 America/Chicago: Withdrawal pending. Tax clearance of 15% ($2,850) required.\n',
    'E05-recovery-call-note.txt': "Victim's note: call 2026-03-09 about 14:00 local from +1 646 555 0108, caller said 'Chainsafe Recovery', asked for $1,500 fee.\n",
}

# ➕ Caso do Seu Antônio (fictício, Operação Isca)
ORIGINAIS_ISCA = {
    'E01-sms-isca.txt': 'SMS recebido de +55 00 00000-0003 em 2026-10-05 07:48 (celular em America/Sao_Paulo): Serra Azul: sua chave Pix foi bloqueada por segurança. Regularize em serraazul-pix.example\n',
    'E02-audio-whatsapp-descricao.txt': "Descrição do áudio exportado (WhatsApp, 'Exportar conversa' com mídia): 14 s, recebido 2026-10-25 21:58 America/Sao_Paulo de +55 00 00000-0021, voz atribuída a 'Cláudio, gerente da agência Centro'. Arquivo de áudio guardado à parte.\n",
    'E03-historico-chamadas.txt': 'Print do histórico de chamadas: chamada recebida de +55 00 00000-0021 em 2026-10-26 08:05 America/Sao_Paulo, duração 23 min.\n',
    'E04-extrato-pix.csv': 'hora_utc,valor_brl,conta_recebedora,chave_destino,id_e2e\n2026-10-26T11:14:22Z,4800.00,L-03,PIXFICT-L03,E2E-FICT-0001\n2026-10-26T11:31:05Z,4950.00,L-05,PIXFICT-L05,E2E-FICT-0002\n',
    'E05-nota-ligacao-recuperacao.txt': "Anotação do cooperado: ligação em 2026-10-28 por volta das 15:00 hora local, de +55 00 00000-0029, 'Dr. Silva, advogado, recupera Pix', pediu R$ 800,00 adiantados de honorários.\n",
    'E06-protocolo-cooperativa.txt': 'Registro interno: cooperado ligou para a central oficial em 2026-10-26 10:20 America/Sao_Paulo; pedido de devolução via MED aberto, protocolo MED-FICT-0001; conta bloqueada para novos Pix.\n',
    'E07-declaracao-rh.txt': 'Declaração do RH (fictícia): Cláudio, gerente da agência Centro, viajou a Belo Horizonte para treinamento presencial de 2026-10-25 18:00 a 2026-10-26 18:00 America/Sao_Paulo, com celular recolhido durante as sessões (lista de presença anexa).\n',
}

def sha256(caminho):
    """Hash do conteúdo em bytes: o mesmo valor que shasum -a 256 / sha256sum mostram."""
    return hashlib.sha256(Path(caminho).read_bytes()).hexdigest()

def coletar(pasta_caso, originais, pasta_orig="originals"):
    """Grava os originais, calcula os hashes da coleta e devolve o registro inicial de custódia."""
    orig = Path(pasta_caso) / pasta_orig
    orig.mkdir(parents=True, exist_ok=True)
    for nome, conteudo in originais.items():
        (orig / nome).write_text(conteudo, encoding="utf-8", newline="\n")   # \n fixo: mesmo hash em qualquer SO
    agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    linhas = [f"{sha256(orig / nome)}  {nome}" for nome in sorted(originais)]
    (Path(pasta_caso) / "hashes-at-collection.txt").write_text("\n".join(linhas) + "\n", encoding="utf-8")
    # Cópia de trabalho: daqui em diante, só ela é aberta.
    shutil.copytree(orig, Path(pasta_caso) / "working", dirs_exist_ok=True)
    return [f"{agora} | coleta de {len(originais)} itens; hashes gravados; cópia para working/"]

CUSTODIA_FICT = coletar("case-FICT-075", ORIGINAIS_FICT)
CUSTODIA_ISCA = coletar("caso-ISCA-075", ORIGINAIS_ISCA, pasta_orig="originais")

print(Path("case-FICT-075/hashes-at-collection.txt").read_text())
print(Path("caso-ISCA-075/hashes-at-collection.txt").read_text())

## 2. Hora local → UTC, com `zoneinfo`

As três perguntas do fluxograma do original (antes do horário de verão? somar quantas horas? passou da meia-noite?) são exatamente o que o banco de fusos do Python responde sozinho. Repare:

- `America/Chicago` muda de UTC-6 para UTC-5 em **8 de março de 2026**;
- `America/Sao_Paulo` fica em UTC-3 o ano todo (sem horário de verão desde 2019).

In [ ]:
from zoneinfo import ZoneInfo        # banco de fusos (Python 3.9+)

UTC = ZoneInfo("UTC")

def para_utc(texto_local, fuso):
    """'2026-01-06 19:42' + 'America/Chicago' → '2026-01-07 01:42' (UTC) e o deslocamento usado."""
    local = datetime.strptime(texto_local, "%Y-%m-%d %H:%M").replace(tzinfo=ZoneInfo(fuso))
    deslocamento = local.utcoffset().total_seconds() / 3600       # -6.0, -5.0, -3.0...
    return local.astimezone(UTC).strftime("%Y-%m-%d %H:%M"), f"UTC{deslocamento:+.0f}"

for t, z in [("2026-01-06 19:42", "America/Chicago"), ("2026-03-07 14:00", "America/Chicago"),
             ("2026-03-09 14:00", "America/Chicago"), ("2026-10-25 21:58", "America/Sao_Paulo")]:
    utc, desl = para_utc(t, z)
    vira = "  ← a data em UTC é o dia seguinte!" if utc[:10] != t[:10] else ""
    print(f"{t} {z:18s} ({desl}) → {utc} UTC{vira}")

## 3. A linha do tempo em UTC

Cada evento tem: ID, hora UTC, carimbo original e fuso, evento, evidência, indicadores exatos e confiança. Os pagamentos já vêm em UTC (E03 do original; E04 do caso BR) e são lidos do próprio arquivo, **da cópia de trabalho**.

In [ ]:
import csv

def ler_pagamentos(caminho, col_hora, col_valor, col_destino):
    """Lê um extrato CSV (já em UTC) da pasta working/."""
    with open(caminho, newline="", encoding="utf-8") as f:
        return [(l[col_hora].replace("T", " ")[:16], float(l[col_valor]), l[col_destino]) for l in csv.DictReader(f)]

# ---------- Caso do original ----------
pag = ler_pagamentos("case-FICT-075/working/E03-exchange-record.csv", "time_utc", "amount_usdt", "to_address")
LT_FICT = []
def ev(lista, original, fuso, evento, evid, indic, conf):
    utc, desl = para_utc(original, fuso) if fuso != "UTC" else (original, "UTC+0")
    lista.append((utc, f"{original} {fuso}", evento, evid, indic, conf))

ev(LT_FICT, "2026-01-06 19:42", "America/Chicago", "Primeiro contato por SMS, 'número errado'", "E01", "+1 202 555 0147", "Confirmado")
ev(LT_FICT, "2026-01-24 08:15", "America/Chicago", "Captura do painel, saldo US$ 1.240,00", "E02", "aurum-desk.example", "Confirmado")
for i, (hora, valor, dest) in enumerate(pag, start=1):
    ev(LT_FICT, hora, "UTC", f"Pagamento {i}: {valor:.0f} USDT", "E03", dest, "Confirmado")
ev(LT_FICT, "2026-03-01 10:05", "America/Chicago", "Exigência de 'imposto' de 15% (US$ 2.850)", "E04", "aurum-desk.example", "Confirmado")
ev(LT_FICT, "2026-03-09 14:00", "America/Chicago", "Ligação de 'Chainsafe Recovery', taxa de US$ 1.500 (hora aproximada)", "E05", "+1 646 555 0108", "Confirmado (anotação da vítima)")

LT_FICT.sort()                                         # ordena por UTC, nunca pela hora local
print(f"{'ID':4s} {'UTC':17s} {'Original':35s} Evento")
for n, (utc, orig, evento, evid, indic, conf) in enumerate(LT_FICT, start=1):
    print(f"T{n:02d}  {utc}  {orig:35s} {evento} [{evid}; {indic}]")

total = sum(v for _, v, _ in pag)
carteiras = sorted({d for _, _, d in pag})
print(f"\nAchado: {total:,.0f} USDT em {len(pag)} pagamentos para {len(carteiras)} carteiras {carteiras} — Confirmado (E03)")

In [ ]:
# ---------- ➕ Caso do Seu Antônio ----------
pix = ler_pagamentos("caso-ISCA-075/working/E04-extrato-pix.csv", "hora_utc", "valor_brl", "conta_recebedora")
SP = "America/Sao_Paulo"
LT_ISCA = []
ev(LT_ISCA, "2026-10-05 07:48", SP, "SMS da campanha 'Pix bloqueado' (não clicou)", "E01", "+55 00 00000-0003; serraazul-pix.example", "Confirmado")
ev(LT_ISCA, "2026-10-25 21:58", SP, "Áudio de WhatsApp com voz atribuída ao gerente Cláudio", "E02", "+55 00 00000-0021", "Confirmado (recebimento)")
ev(LT_ISCA, "2026-10-26 08:05", SP, "Início da ligação de 23 min do 'Cláudio'", "E03", "+55 00 00000-0021", "Confirmado")
for i, (hora, valor, conta) in enumerate(pix, start=1):
    ev(LT_ISCA, hora, "UTC", f"Pix {i}: R$ {valor:,.2f}", "E04", conta, "Confirmado")
ev(LT_ISCA, "2026-10-26 10:20", SP, "Cooperado liga para a central oficial; pedido de devolução aberto (MED-FICT-0001)", "E06", "MED-FICT-0001", "Confirmado")
ev(LT_ISCA, "2026-10-28 15:00", SP, "Ligação do 'Dr. Silva', pede R$ 800 adiantados (hora aproximada)", "E05", "+55 00 00000-0029", "Confirmado (anotação da vítima)")
LT_ISCA.sort()

for n, (utc, orig, evento, evid, indic, conf) in enumerate(LT_ISCA, start=1):
    print(f"T{n:02d}  {utc}  {orig:35s} {evento} [{evid}; {indic}]")

# A pergunta da Prática BR: os Pix caem dentro da ligação?
inicio = datetime.strptime(para_utc("2026-10-26 08:05", SP)[0], "%Y-%m-%d %H:%M")
from datetime import timedelta
fim = inicio + timedelta(minutes=23)                    # timedelta: funciona até virando a hora
print(f"\nLigação: {inicio:%H:%M}–{fim:%H:%M} UTC")
for hora, valor, conta in pix:
    h = datetime.strptime(hora, "%Y-%m-%d %H:%M")
    dentro = inicio <= h <= fim
    print(f"  Pix {hora[11:]} UTC → {'dentro' if dentro else f'{(h - fim).seconds // 60} min DEPOIS do fim'} da ligação")
print("Se algum caiu fora: achado ou problema de relógio/arredondamento? Escreva a sua resposta no relatório.")

print(f"\nAchado: R$ {sum(v for _, v, _ in pix):,.2f} em {len(pix)} Pix para {len({c for *_, c in pix})} contas "
      f"{sorted({c for *_, c in pix})} — Confirmado (E04)")

## 4. A checagem de custódia (e o que fazer quando falha)

Primeiro, a checagem normal (o `shasum -a 256 -c` do original, em Python). Depois, **estragamos de propósito** um original do caso BR (como se alguém tivesse "corrigido" o extrato no editor) e rodamos de novo.

Regra do original: a falha vai para o **histórico de custódia**. O `hashes-at-collection.txt` não é recalculado.

In [ ]:
def conferir(pasta_caso, pasta_orig, custodia, quem="analista júnior (Vigia)"):
    """Equivale a: cd originals && shasum -a 256 -c ../hashes-at-collection.txt"""
    ok_geral = True
    for linha in (Path(pasta_caso) / "hashes-at-collection.txt").read_text().splitlines():
        esperado, nome = linha.split(maxsplit=1)
        atual = sha256(Path(pasta_caso) / pasta_orig / nome)
        ok = (atual == esperado)
        ok_geral &= ok
        print(f"  {nome}: {'OK' if ok else 'FAILED'}")
        if not ok:                                      # registra a falha; NÃO recalcula a coleta
            custodia.append(f"{datetime.now(timezone.utc):%Y-%m-%dT%H:%M:%SZ} | FALHA de hash em {nome}: "
                            f"esperado {esperado[:12]}…, obtido {atual[:12]}… | checado por {quem}")
    if ok_geral:
        custodia.append(f"{datetime.now(timezone.utc):%Y-%m-%dT%H:%M:%SZ} | checagem de hash: todos OK | por {quem}")
    return ok_geral

print("case-FICT-075:"); conferir("case-FICT-075", "originals", CUSTODIA_FICT)
print("caso-ISCA-075:"); conferir("caso-ISCA-075", "originais", CUSTODIA_ISCA)

# 💥 O acidente: alguém "arruma" um valor no original do extrato.
alvo = Path("caso-ISCA-075/originais/E04-extrato-pix.csv")
alvo.write_text(alvo.read_text(encoding="utf-8").replace("4950.00", "4900.00"), encoding="utf-8")
hash_coleta_antes = Path("caso-ISCA-075/hashes-at-collection.txt").read_text()

print("\ncaso-ISCA-075 depois do acidente:"); conferir("caso-ISCA-075", "originais", CUSTODIA_ISCA)
assert Path("caso-ISCA-075/hashes-at-collection.txt").read_text() == hash_coleta_antes   # a coleta ficou intacta
print("\nHistórico de custódia do caso BR:")
for linha in CUSTODIA_ISCA:
    print("  ", linha)
print("\nO hashes-at-collection.txt NÃO foi recalculado. É isso que permite provar o que mudou e quando.")

## 5. 🎮 Quiz: confirmado, provável ou contestado?

Responda **c** (confirmado: um item de evidência mostra diretamente), **p** (provável: há apoio, mas indireto ou de uma fonte só) ou **x** (contestado ou sem sustentação). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("🇺🇸 US$ 19.000 em cinco pagamentos para duas carteiras", "c", "O registro da corretora (E03) mostra diretamente."),
    ("🇺🇸 A exigência do 'imposto' de 15% era golpe", "cp", "E04 mostra a exigência; que é golpe vem da combinação com o padrão (taxa para sacar). Forte, mas é inferência: aceitamos 'confirmado' (se você citar E04 e o padrão) ou 'provável'."),
    ("🇺🇸 A 'Chainsafe Recovery' é o mesmo grupo dos golpistas", "x", "Nada liga +1 646 555 0108 aos outros indicadores; listas de vítimas são vendidas."),
    ("🇧🇷 R$ 9.750,00 em dois Pix para as contas L-03 e L-05", "c", "O extrato (E04) mostra diretamente."),
    ("🇧🇷 A voz do áudio não era do Cláudio", "p", "E07 prova onde o Cláudio estava e que o celular estava recolhido; não prova quem gravou. Provável, com o motivo escrito."),
    ("🇧🇷 O número +55 00 00000-0021 pertence ao mesmo grupo do SMS de 5/out", "x", "Neste lote, nenhuma evidência liga os dois. O grafo (Dia 76) pode mudar isso."),
    ("🇧🇷 O 'Dr. Silva' é da mesma quadrilha", "x", "Encaixa na história, mas não há indicador em comum. Viés retrospectivo em ação."),
    ("🇧🇷 O cooperado abriu o pedido de devolução às 13:20 UTC de 26/out", "c", "E06 registra 10:20 em America/Sao_Paulo = 13:20 UTC."),
]

acertos = 0
for i, (achado, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {achado}")
    if MODO_JOGO:
        r = ""
        for _ in range(5):                             # nunca trava
            r = input("   c / p / x? ").strip().lower()[:1]
            if r in ("c", "p", "x"):
                break
        ok = r in certo                                 # "cp" = as duas leituras são defensáveis
        acertos += ok
        print("   ✅" if ok else f"   ❌ era {' ou '.join(certo)}")
    print("   →", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 6. 💾 Exportar a planilha (três abas em três CSV)

Registro de evidências, linha do tempo e achados, do caso BR, prontos para abrir no Google Planilhas ou no Excel.

In [ ]:
#@title Exportar o caso BR { display-mode: "form" }
coletado_por = "Analista júnior (Vigia), com o cooperado exportando no próprio celular"  #@param {type:"string"}
local_de_guarda = "Pasta cifrada do caso na Vigia; áudio .opus em mídia separada, acesso só Rafa e analista"  #@param {type:"string"}

pasta = Path("notas/caso-ISCA-075"); pasta.mkdir(parents=True, exist_ok=True)
coleta = {l.split(maxsplit=1)[1]: l.split()[0] for l in Path("caso-ISCA-075/hashes-at-collection.txt").read_text().splitlines()}

with open(pasta / "registro-evidencias.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["item", "arquivo", "fonte", "coletado_por", "sha256_coleta", "guarda", "historico_custodia"])
    for nome, h in coleta.items():
        fonte = "cooperativa" if nome[:3] in ("E06", "E07") else "cooperado (celular/extrato)"
        w.writerow([nome[:3], nome, fonte, coletado_por, h, local_de_guarda, " || ".join(CUSTODIA_ISCA)])

with open(pasta / "linha-do-tempo-utc.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "hora_utc", "original_e_fuso", "evento", "evidencia", "indicadores", "confianca"])
    for n, linha in enumerate(LT_ISCA, start=1):
        w.writerow([f"T{n:02d}", *linha])

ACHADOS = [
    ("R$ 9.750,00 em 2 Pix para 2 contas recebedoras (L-03, L-05)", "confirmado", "E04"),
    ("A voz do áudio não era do Cláudio", "provável", "E07 (onde ele estava; não prova quem gravou)"),
    ("+55 00 00000-0021 ligado à campanha do SMS de 5/out", "contestado/sem sustentação", "nenhum indicador em comum neste lote"),
    ("'Dr. Silva' (+55 00 00000-0029) é da mesma quadrilha", "contestado/sem sustentação", "nenhum indicador em comum; listas de vítimas circulam"),
]
with open(pasta / "achados.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["achado", "confianca", "evidencia"])
    w.writerows(ACHADOS)

for arq in sorted(pasta.iterdir()):
    print(f"💾 {arq}  SHA-256 {sha256(arq)[:16]}…")
try:
    from google.colab import files
    for arq in sorted(pasta.iterdir()):
        files.download(str(arq))
except ImportError:
    print("(Fora do Colab: ficou em notas/caso-ISCA-075/)")

## ✅ Antes de fechar o Dia 75

- [ ] `case-FICT-075/` com originals, working, hashes e as três abas (≥ 9 eventos)
- [ ] US$ 19.000, cinco pagamentos, duas carteiras: confirmado, citando E03
- [ ] T01 com data UTC no dia seguinte; a ligação de 9/mar em UTC-5
- [ ] `caso-ISCA-075/` com o mesmo conjunto; R$ 9.750,00, duas contas, confirmado, citando E04
- [ ] A falha de hash ficou no histórico de custódia, e a coleta não foi recalculada

**Amanhã (Dia 76):** o pacote para a delegacia, o pedido no banco, o canal da plataforma, e o Seu Antônio entra no grafo da quadrilha. A Fase 5 fecha.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>